# 05 · Analysis: figures, statistics, Grad-CAM
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/05_analysis.ipynb)

**Runtime → T4 GPU → Run all.** Использует сохранённые test-предсказания из `04_evaluate`
(`MyDrive/bgbias/results/test/`) и checkpoints из `MyDrive/bgbias/runs/`. Ничего не обучает и не подбирает.

Результат: рисунки в `report/figures/`, таблицы в `results/analysis/` — всё автоматически пушится в GitHub.

1. Точность по условиям (A vs B)
2. Ablation по p
3. Bootstrap-интервал для разницы gap(A) − gap(B)
4. Какие породы сильнее зависят от фона; кошки vs собаки
5. Примеры: A ошибается после замены фона, B — нет; Grad-CAM
6. Grad-CAM в цифрах: доля внимания на животном

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
REPO_URL = "https://github.com/mundmina/dl_project.git"
REPO_DIR = "/content/dl_project"
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:  # always run the latest code from GitHub
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    from google.colab import userdata
    userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from bgbias import analysis as A, evaluate as E
from bgbias.gradcam import GradCAM, eval_mask_transform, foreground_share
from bgbias.models import load_checkpoint
from bgbias.project import make_dataset

SPLIT = "test"
PRED_DIR = DRIVE_ROOT / "results" / SPLIT
FIG_DIR = Path(REPO_DIR) / "report" / "figures"
TAB_DIR = Path(REPO_DIR) / "results" / "analysis"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda")

# Fixed identity colours (validated palette: first three categorical slots).
COLORS = {"resnet_A": "#2a78d6", "resnet_B": "#eb6834", "hog_svm": "#1baf7a"}
LABELS = {"resnet_A": "ResNet-18 A (original images)", "resnet_B": "ResNet-18 B (background swap, p = 0.5)",
          "hog_svm": "HOG + SVM"}
INK, INK2, RULE, SURFACE = "#0b0b0b", "#52514e", "#d6d5d0", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": RULE, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#ebeae6", "grid.linewidth": 0.8,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "font.size": 10, "lines.linewidth": 2, "legend.frameon": False,
})

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

tables = {p.name.removesuffix("_predictions.csv"): pd.read_csv(p) for p in sorted(PRED_DIR.glob("*_predictions.csv"))}
per_run = pd.DataFrame({run: E.summarize(t) for run, t in tables.items()}).T
per_run["model"] = [A.model_name(r) for r in per_run.index]
per_run.to_csv(TAB_DIR / "summary_per_run.csv")
print(sorted(tables))

## 1. Точность по условиям

In [ ]:
CONDS = [("original", "Original"), ("swapped", "Background swapped"), ("fgonly", "Foreground only")]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for model, offset in (("resnet_A", -0.12), ("resnet_B", 0.12)):
    runs = per_run[per_run.model == model]
    mean = np.array([runs[f"acc_{c}"].mean() for c, _ in CONDS])
    std = np.array([runs[f"acc_{c}"].std() for c, _ in CONDS])
    x = np.arange(len(CONDS)) + offset
    ax.errorbar(x, mean, yerr=std, fmt="o", ms=8, color=COLORS[model], ecolor=COLORS[model],
                elinewidth=1.5, capsize=3, label=LABELS[model])
    for xi, v in zip(x, mean):
        ax.annotate(f"{v:.1%}", (xi, v), xytext=(10 if offset > 0 else -10, 0), textcoords="offset points",
                    ha="left" if offset > 0 else "right", va="center", fontsize=8, color=INK2)
ax.set_xticks(range(len(CONDS)), [label for _, label in CONDS])
ax.set_xlim(-0.6, len(CONDS) - 0.4)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
ax.set_ylabel("Test accuracy (mean ± std, 3 seeds)")
ax.set_title("Background swap hurts model A; model B is almost unaffected")
ax.legend(loc="lower left")
save(fig, "fig1_accuracy_by_condition")

## 2. Ablation: вероятность замены фона p

In [ ]:
abl = per_run[per_run.model.isin(A.P_SWAP)].copy()
abl["p"] = abl.model.map(A.P_SWAP)
g = abl.groupby("p")[["acc_original", "acc_swapped", "gap_swap"]].agg(["mean", "std"])
g.to_csv(TAB_DIR / "ablation_p.csv")
print("p values available:", list(g.index))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
panels = [("acc_original", "Accuracy on original images"), ("gap_swap", "Robustness gap: original − swapped")]
for ax, (metric, title) in zip(axes, panels):
    ax.errorbar(g.index, g[(metric, "mean")], yerr=g[(metric, "std")], fmt="-o", ms=8, color=INK2, capsize=3)
    for p, v in zip(g.index, g[(metric, "mean")]):
        tag = {0.0: "A", 0.5: "B"}.get(p)
        if tag:
            ax.annotate(tag, (p, v), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=9, color=INK)
    ax.set_xticks(g.index)
    ax.set_xlabel("Probability of background swap during training")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
    ax.set_title(title)
axes[1].set_ylim(bottom=min(0, axes[1].get_ylim()[0]))
fig.tight_layout()
save(fig, "fig2_ablation_p")
g.round(4)

## 3. Статистика: bootstrap по тестовым изображениям

In [ ]:
stats = A.bootstrap_gap_difference(tables, "resnet_A", "resnet_B", n_boot=2000)
(TAB_DIR / "bootstrap_gap_difference.json").write_text(json.dumps(stats, indent=1))
print(f"gap A = {stats['gap_A']:.2%}, gap B = {stats['gap_B']:.2%}")
print(f"gap A - gap B = {stats['difference']:.2%}, 95% CI [{stats['ci_low']:.2%}, {stats['ci_high']:.2%}], "
      f"share of bootstrap samples <= 0: {stats['p_le_0']:.4f}")

## 4. Какие породы зависят от фона сильнее всего

In [ ]:
gap_a, gap_b = A.per_breed_gap(tables, "resnet_A"), A.per_breed_gap(tables, "resnet_B")
breeds = pd.DataFrame({"gap_A": gap_a, "gap_B": gap_b}).sort_values("gap_A")
species = tables[next(iter(tables))].groupby("breed").species.first()
breeds["species"] = species
breeds.to_csv(TAB_DIR / "per_breed_gap.csv")

fig, ax = plt.subplots(figsize=(7.5, 10))
y = np.arange(len(breeds))
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True, color="#ebeae6")
ax.hlines(y, breeds.gap_B, breeds.gap_A, color=RULE, lw=2, zorder=1)
for model, col in (("resnet_A", "gap_A"), ("resnet_B", "gap_B")):
    ax.plot(breeds[col], y, "o", ms=7, color=COLORS[model], label=LABELS[model], zorder=2,
            markeredgecolor=SURFACE, markeredgewidth=1.5)
ax.axvline(0, color=INK2, lw=0.8)
ax.set_yticks(y, [f"{b.replace('_', ' ')} ({s})" for b, s in zip(breeds.index, breeds.species)], fontsize=8)
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_xlabel("Accuracy drop when the background is swapped (mean of 3 seeds)")
ax.set_title("Per-breed background dependence")
ax.legend(loc="upper left")   # top-left is empty: the most affected breeds sit at the top right
save(fig, "fig3_per_breed_gap")

print(breeds.groupby("species")[["gap_A", "gap_B"]].mean().round(4))
breeds.sort_values("gap_A", ascending=False).head(8).round(3)

## 5. Примеры: A ошибается после замены фона, B — нет (Grad-CAM, seed 0)

In [ ]:
idx, n_total = A.flip_examples(tables, "resnet_A", "resnet_B", n=6, seed=0)
print(f"{n_total} test images: every A seed is right on original and wrong on swapped, every B seed right on both")

breed_of = dict(zip(data.splits[SPLIT].label, data.splits[SPLIT].breed))
models = {m: load_checkpoint(RUNS / f"{m}_seed0" / "best.pt", data.num_classes, device)[0]
          for m in ("resnet_A", "resnet_B")}
views = {c: make_dataset(data, SPLIT, condition=c) for c in ("original", "swapped")}
geometry = eval_mask_transform()

def cam_overlay(img, cam):
    """Unattended regions are darkened; attended regions keep full brightness."""
    arr = np.asarray(geometry(img), dtype=np.float32) / 255
    return (arr * (0.15 + 0.85 * cam[..., None])).clip(0, 1)

cols = [("original", None), ("original", "resnet_A"), ("original", "resnet_B"),
        ("swapped", None), ("swapped", "resnet_A"), ("swapped", "resnet_B")]
fig, axes = plt.subplots(len(idx), len(cols), figsize=(2.3 * len(cols), 2.5 * len(idx)), squeeze=False)
for c in ("original", "swapped"):
    x = torch.stack([views[c][i][0] for i in idx]).to(device)
    for m, model in models.items():
        cam = GradCAM(model)
        maps, preds = cam(x)
        cam.remove()
        for r, i in enumerate(idx):
            j = cols.index((c, m))
            ax = axes[r, j]
            ax.imshow(cam_overlay(views[c].render(i), maps[r].cpu().numpy()))
            ok = preds[r].item() == data.splits[SPLIT].label.iat[i]
            ax.set_title(f"{'A' if m == 'resnet_A' else 'B'}: {breed_of[preds[r].item()].replace('_', ' ')}"
                         f" {'✓' if ok else '✗'}", fontsize=8, fontweight="normal",
                         color=INK if ok else "#d03b3b", loc="center")
for r, i in enumerate(idx):
    for c in ("original", "swapped"):
        ax = axes[r, cols.index((c, None))]
        ax.imshow(geometry(views[c].render(i)))
        ax.set_title(f"{c}\ntrue: {data.splits[SPLIT].breed.iat[i].replace('_', ' ')}",
                     fontsize=8, fontweight="normal", loc="center")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
save(fig, "fig4_flip_examples_gradcam")

## 6. Grad-CAM в цифрах

Доля «внимания» Grad-CAM, попадающая на животное (по trimap-маске), на 500 случайных тестовых изображениях,
все 3 seed каждой модели. Для сравнения — доля площади кадра, которую занимает животное.
Grad-CAM на рисунке выше — иллюстрация; вывод делаем по этим цифрам.

In [ ]:
rng = np.random.default_rng(0)
sample = np.sort(rng.choice(len(data.splits[SPLIT]), size=500, replace=False))
rows = []
for model_name in ("resnet_A", "resnet_B"):
    for seed in (0, 1, 2):
        model, cfg = load_checkpoint(RUNS / f"{model_name}_seed{seed}" / "best.pt", data.num_classes, device)
        for c in ("original", "swapped"):
            share, area = foreground_share(model, data, SPLIT, c, sample, device)
            rows.append({"model": model_name, "seed": seed, "condition": c,
                         "cam_on_pet": share.mean(), "pet_area": area.mean()})
cam_stats = pd.DataFrame(rows)
cam_stats.to_csv(TAB_DIR / "gradcam_foreground_share.csv", index=False)
cam_stats.groupby(["model", "condition"])[["cam_on_pet", "pet_area"]].agg(["mean", "std"]).round(3)

## Сохранение и push

In [ ]:
import shutil
from bgbias.github_sync import sync

shutil.copytree(FIG_DIR, DRIVE_ROOT / "figures", dirs_exist_ok=True)
shutil.copytree(TAB_DIR, DRIVE_ROOT / "results" / "analysis", dirs_exist_ok=True)
sync(REPO_DIR, "Analysis: figures, bootstrap, per-breed gaps, Grad-CAM")